# Positional Encoding

## What is Positional Encoding?

Transformers process all tokens in parallel, so **self-attention does not inherently know the order of tokens**.

Positional Encoding adds information about **where each token appears in the sequence**.

```text
Token Embedding + Positional Encoding
                ↓
        Position-aware input
                ↓
        Multi-Head Attention
```

---

## Analogy

Imagine reading:

> `The dog chased the cat`

The words have meaning, but their **order** also matters.

Positional Encoding gives every token a positional identity:

```text
The → Position 0
dog → Position 1
chased → Position 2
the → Position 3
cat → Position 4
```

So the model receives both:

* **What the token means**
* **Where the token is**


In [16]:
import torch
import math
from torch import nn

In [17]:
class Encoding(nn.Module):
    def __init__(self,maximum_size,feature_size):
        super().__init__()
        encoding=torch.zeros(maximum_size,feature_size)
        print(encoding.shape)
        position=torch.arange(maximum_size).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,feature_size,2)*(-math.log(10000.0)/feature_size))
        encoding[:,0::2]=torch.sin(position*div_term)
        encoding[:,1::2]=torch.cos(position*div_term)
        self.register_buffer("encoding",encoding)
    def forward(self,x):
        size=x.shape[1]
        return self.encoding[:size]+x

In [25]:
model=Encoding(5,2)
print(model.encoding)
print(model.encoding.shape)
embedding=nn.Embedding(10,2)
prompt=torch.tensor([[1,5,8,9,6],[6,5,2,3,4]])
x=embedding(prompt)
print(f"\n\n{model(x)}")

torch.Size([5, 2])
tensor([[ 0.0000,  1.0000],
        [ 0.8415,  0.5403],
        [ 0.9093, -0.4161],
        [ 0.1411, -0.9900],
        [-0.7568, -0.6536]])
torch.Size([5, 2])


tensor([[[ 1.3069,  2.1805],
         [-0.1566,  1.8046],
         [ 1.0895, -0.8492],
         [ 1.5029, -0.3511],
         [-2.2003, -1.8583]],

        [[-1.4435, -0.2047],
         [-0.1566,  1.8046],
         [ 1.1028, -0.8321],
         [ 0.6254, -0.5655],
         [-0.9185, -0.4026]]], grad_fn=<AddBackward0>)



---

## Input Size vs Sequence Length

If:

```text
input_size = 8
sequence_length = 5
```

then every token is represented using **8 features**, and there are **5 tokens**.

Therefore:

```text
Input shape = [5, 8]
```

For a batch:

```text
Input shape = [batch, sequence, features]
            = [2, 5, 8]
```

### `input_size`

Number of features used to represent each token.

### `maximum_size`

Maximum number of positions for which positional encodings are created.

For:

```python
Encoding(8, 100)
```

the positional encoding matrix has:

```text
[100, 8]
```

meaning:

```text
100 positions × 8 features
```

---

## Positional Encoding Matrix

Unlike the attention score matrix, positional encoding is **not a similarity matrix**.

For 5 tokens and 8 features:

```text
Positional Encoding

             Features
           1 2 3 4 5 6 7 8
Position 0 [................]
Position 1 [................]
Position 2 [................]
Position 3 [................]
Position 4 [................]
```

Its shape is:

```text
[sequence_length, input_size]
```

While attention produces:

```text
[sequence_length, sequence_length]
```

because attention measures relationships between tokens.

---

## Sinusoidal Positional Encoding

The original Transformer uses sine and cosine functions.

For even dimensions:

$$
PE(pos,2i)=
\sin\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

For odd dimensions:

$$
PE(pos,2i+1)=
\cos\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

Each position therefore receives a unique numerical pattern.

---

## Adding Position Information

The positional encoding is added directly to the token representation:

$$
X_{position}=X+PE
$$

For example:

```text
Token representation
[0.2, 0.7, 0.4, 0.1]

        +

Position encoding
[0.8, 0.5, 0.1, 0.9]

        ↓

Position-aware representation
[1.0, 1.2, 0.5, 1.0]
```

The shape does **not** change.

```text
X   → [batch, sequence, features]
PE  → [sequence, features]

X + PE → [batch, sequence, features]
```

---
---

## Why `torch.sin()` instead of `math.sin()`?

`position * div_term` is a **tensor containing many values**.

```python
torch.sin(tensor)
```

applies sine element-by-element.

`math.sin()` expects a single Python number.

---

## `register_buffer()`

```python
self.register_buffer("encoding", encoding)
```

The positional encoding is **not a learnable parameter**.

`register_buffer()` tells PyTorch:

> Keep this tensor as part of the model, but don't update it during training.

It will still move with the model when using:

```python
model.to(device)
```

---

## Position Encoding vs Attention Matrix

|            | Positional Encoding    | Attention Scores               |
| ---------- | ---------------------- | ------------------------------ |
| Purpose    | Represents position    | Represents token relationships |
| Shape      | `sequence × features`  | `sequence × sequence`          |
| Example    | `5 × 8`                | `5 × 5`                        |
| Learnable? | Sinusoidal version: No | Q/K projections are learned    |
| Used when  | Before attention       | During attention               |

### Core idea

> **Positional Encoding tells the Transformer where a token is; Attention determines how tokens interact with each other.**

### Transformer flow

```text
Token Embeddings
       ↓
Positional Encoding
       ↓
Multi-Head Attention
       ↓
Residual Connection
       ↓
LayerNorm
       ↓
Feed-Forward Network
       ↓
Residual Connection
       ↓
LayerNorm
```
